In [0]:
from pyspark.sql import functions as F

PERF_TABLE = "sentinel_dev.gold.fact_orders_perf"

In [0]:
perf_df = (
    spark.range(0, 5_000_000)

    .withColumn(
        "order_id",
        F.concat(
            F.lit("PERF-"),
            F.col("id").cast("string")
        )
    )

    .withColumn(
        "customer_key",
        (F.col("id") % 100_000).cast("long")
    )

    .withColumn(
        "product_key",
        (F.col("id") % 10_000).cast("long")
    )

    .withColumn(
        "order_date",
        F.date_sub(
            F.current_date(),
            (F.col("id") % 365).cast("int")
        )
    )

    .withColumn(
        "quantity",
        ((F.col("id") % 5) + 1).cast("int")
    )

    .withColumn(
        "unit_price",
        (
            ((F.col("id") % 5000) + 500)
            / 100
        ).cast("decimal(18,2)")
    )

    .withColumn(
        "total_amount",
        (
            F.col("quantity")
            * F.col("unit_price")
        ).cast("decimal(18,2)")
    )

    .drop("id")
)

In [0]:
print(perf_df.count())

display(perf_df.limit(10))

In [0]:
(
    perf_df
        .repartition(200)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(PERF_TABLE)
)

print("Performance dataset created.")

In [0]:
%sql
DESCRIBE DETAIL sentinel_dev.gold.fact_orders_perf;